# OOP => Class Patterns With `match`

`match` can test the **type and attributes** of an object in one pattern. This builds on the `match` / `case` notebook in Control Flow.

| Pattern | Example | Matches |
|---|---|---|
| Class | `case Point():` | Any `Point` instance |
| Keyword attributes | `case Point(x=0, y=0):` | A `Point` whose attributes have those values |
| Positional attributes | `case Point(0, 0):` | Same, using `__match_args__` |
| Capture attributes | `case Point(x=px):` | A `Point`, and binds `x` to `px` |
| Dotted constant | `case Color.RED:` | An enum member or other constant |
| Nested | `case Line(Point(0, 0), end):` | Patterns inside patterns |
| Type check | `case int() \| float():` | Any number type |

---

## Class Patterns

```python
match shape:
    case Circle(radius=r):
        ...
    case Rectangle(width=w, height=h):
        ...
```

* The class name is followed by parentheses, even when no attributes are matched: `case str():`.
* **Keyword form** (`Point(x=0)`) reads attributes by name.
* **Positional form** (`Point(0, 0)`) needs `__match_args__`, which lists the attribute names in order.
* Dataclasses create `__match_args__` automatically.

### Custom `__match_args__`

```python
class Point:
    __match_args__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y
```

---

## Constants With Dotted Names

A bare name is a capture. A **dotted name** is compared by value:

```python
match color:
    case Color.RED:
        ...
    case Color.GREEN:
        ...
```

---

## Nested Patterns and Guards

Patterns can be combined: an OR pattern between classes, a class inside a sequence, and an `if` guard.

```python
match event:
    case Click(position=(x, y)) if x < 0:
        ...
```

---

## Important

* A class pattern uses `isinstance()`, so **subclasses match** too.
* Order matters: put specific patterns before general ones.
* Use `case _:` (or `raise`) to handle unexpected types.

## Source

https://docs.python.org/3/reference/compound_stmts.html#class-patterns

https://peps.python.org/pep-0636/

In [ ]:
from dataclasses import dataclass
from enum import Enum

@dataclass
class Point:
    x: float
    y: float

@dataclass
class Circle:
    center: Point
    radius: float

@dataclass
class Rectangle:
    width: float
    height: float

# Keyword and positional patterns (dataclasses provide __match_args__)
def describe(shape):
    match shape:
        case Circle(center=Point(x=0, y=0), radius=r):
            return f"circle at the origin, radius {r}"
        case Circle(radius=r):
            return f"circle, radius {r}"
        case Rectangle(w, h) if w == h:
            return f"square, side {w}"
        case Rectangle(w, h):
            return f"rectangle {w} x {h}"
        case _:
            return "unknown shape"

for shape in (Circle(Point(0, 0), 2), Circle(Point(1, 1), 3), Rectangle(2, 2), Rectangle(3, 4), "text"):
    print(describe(shape))

# Type checks with class patterns
for value in (3, 2.5, "hi", None):
    match value:
        case int() | float():
            print(value, "is a number")
        case str():
            print(value, "is a string")
        case _:
            print(value, "is something else")

# A custom __match_args__
class Vector:
    __match_args__ = ("dx", "dy")

    def __init__(self, dx, dy):
        self.dx, self.dy = dx, dy

match Vector(0, 5):
    case Vector(0, dy):
        print("vertical vector of length", dy)

# Dotted names match constants; a bare name would capture instead
class Color(Enum):
    RED = 1
    GREEN = 2

for color in Color:
    match color:
        case Color.RED:
            print(color.name, "-> stop")
        case Color.GREEN:
            print(color.name, "-> go")

# Subclasses match their parent's class pattern
class Square(Rectangle):
    pass

match Square(4, 4):
    case Rectangle(w, h):
        print("a Square matches Rectangle(...):", w, h)